In [1]:
import numpy as np
import scipy.stats as st

np.random.seed(52)

def inv_pareto(u, alpha):
    return (1 - u) ** (1 / (1 - alpha))

def med(alpha):
    return 2 ** (1 / (alpha - 1))

n = 100
beta = 0.95
theta = 52

x = inv_pareto(st.uniform(loc=0, scale=1).rvs(size=n), alpha=theta)

theta_hat = 1 + 1 / np.mean(np.log(x))
m_true = med(theta)
m_hat = med(theta_hat)

z1 = st.norm.ppf((1 - beta) / 2)
z2 = st.norm.ppf((1 + beta) / 2)

m_left = m_hat - m_hat * np.log(2) * z2 / (np.sqrt(n) * (theta_hat - 1))
m_right = m_hat - m_hat * np.log(2) * z1 / (np.sqrt(n) * (theta_hat - 1))
m_len = m_right - m_left

t_left = theta_hat - z2 / (np.sum(np.log(x)) / np.sqrt(n))
t_right = theta_hat - z1 / (np.sum(np.log(x)) / np.sqrt(n))
t_len = t_right - t_left

B1 = 1000
d_np = []
for _ in range(B1):
    xb = np.random.choice(x, size=n, replace=True)
    d_np.append(1 + 1 / np.mean(np.log(xb)) - theta_hat)
d_np = np.array(sorted(d_np))

k1 = int((1 - beta) / 2 * B1 - 1)
k2 = int((1 + beta) / 2 * B1 - 1)

np_left = theta_hat - d_np[k2]
np_right = theta_hat - d_np[k1]
np_len = np_right - np_left

B2 = 50000
d_p = []
for _ in range(B2):
    xb = inv_pareto(st.uniform(loc=0, scale=1).rvs(size=n), alpha=theta_hat)
    d_p.append(1 + 1 / np.mean(np.log(xb)) - theta_hat)
d_p = np.array(sorted(d_p))

p_left = theta_hat - d_p[k2]
p_right = theta_hat - d_p[k1]
p_len = p_right - p_left

ans = [
    ("Асимптотический для параметра", t_len),
    ("Непараметрический бутстрап", np_len),
    ("Параметрический бутстрап", p_len)
]
ans.sort(key=lambda item: item[1])

print("Истинное значение параметра:", theta)
print("Оценка параметра:", theta_hat)
print("")

print("Истинная медиана:", m_true)
print("Оценка медианы:", m_hat)
print("Асимптотический доверительный интервал для медианы:")
print(f"{m_left} < m < {m_right}")
print("Длина интервала:", m_len)
print("")

print("Асимптотический доверительный интервал для параметра:")
print(f"{t_left} < theta < {t_right}")
print("Длина интервала:", t_len)
print("")

print("Непараметрический бутстраповский доверительный интервал для параметра:")
print(f"{np_left} < theta < {np_right}")
print("Длина интервала:", np_len)
print("")

print("Параметрический бутстраповский доверительный интервал для параметра:")
print(f"{p_left} < theta < {p_right}")
print("Длина интервала:", p_len)
print("")

print("Сравнение интервалов для параметра по длине:")
for i, (name, val) in enumerate(ans, 1):
    print(f"{i}. {name}: {round(val, 3)}")


Истинное значение параметра: 52
Оценка параметра: 45.93946013047142

Истинная медиана: 1.0136839003226856
Оценка медианы: 1.0155435852263073
Асимптотический доверительный интервал для медианы:
1.012473543568693 < m < 1.0186136268839217
Длина интервала: 0.00614008331522875

Асимптотический доверительный интервал для параметра:
37.13148779643165 < theta < 54.747432464511185
Длина интервала: 17.615944668079536

Непараметрический бутстраповский доверительный интервал для параметра:
36.77552069289992 < theta < 52.9717225768478
Длина интервала: 16.196201883947886

Параметрический бутстраповский доверительный интервал для параметра:
54.03238128936185 < theta < 58.29148293655252
Длина интервала: 4.259101647190668

Сравнение интервалов для параметра по длине:
1. Параметрический бутстрап: 4.259
2. Непараметрический бутстрап: 16.196
3. Асимптотический для параметра: 17.616
